# 07 · Solar Panel Detection

Map photovoltaic installations from aerial and satellite imagery.

## Contents
1. Data acquisition
2. Solar panel detection with GeoAI
3. Capacity estimation
4. Time-series monitoring
5. Area calculation and visualisation

## 1 · Data Acquisition

In [ ]:
import pygeovision as pgv
from pathlib import Path

client = pgv.PyGeoVision()

# NAIP (0.6m, best for rooftop solar) or Sentinel-2 (10m, utility-scale)
results = client.search(
    bbox=(-119.0, 36.6, -118.9, 36.7),   # Central Valley, California
    date_range=("2022-01-01", "2024-12-31"),
    collections=["naip"],
    providers=["planetary_computer"],
    max_results=3,
)

if not results:
    results = client.search(
        bbox=(-119.0, 36.6, -118.9, 36.7),
        date_range=("2024-01-01", "2024-12-31"),
        providers=["planetary_computer"],
        cloud_cover_max=5,
        max_results=3,
    )

downloads = client.download(results[:1], output_dir="./data/solar/",
                             post_process=["reproject:EPSG:4326", "cog"])
img_path = downloads[0].path
print(f"Image: {img_path}")

Image: data/solar/m_3611948_se_11_060_20230821.tif


## 2 · Detection with GeoAI

In [ ]:
# Solar panel segmentation using GeoAI
client.geoai.segment.solar_panels(
    img_path,
    output_path="./results/solar/solar_mask.tif",
    output_vector="./results/solar/solar_panels.geojson",
    confidence_threshold=0.6,
)
print("Solar panel detection complete")

Solar panel detection complete


## 3 · Capacity Estimation

In [ ]:
import geopandas as gpd
import numpy as np

gdf = gpd.read_file("./results/solar/solar_panels.geojson")
gdf_utm = gdf.to_crs("EPSG:32611")     # UTM Zone 11N for California
gdf["area_m2"] = gdf_utm.geometry.area

# Typical solar panel efficiency: ~200W per m²
# Average capacity factor for utility-scale: ~25%
EFFICIENCY_W_PER_M2 = 200
CAPACITY_FACTOR = 0.25

gdf["capacity_kw"] = gdf["area_m2"] * EFFICIENCY_W_PER_M2 / 1000
gdf["annual_kwh"] = gdf["capacity_kw"] * 8760 * CAPACITY_FACTOR

total_area_ha = gdf["area_m2"].sum() / 10000
total_capacity_mw = gdf["capacity_kw"].sum() / 1000
annual_gwh = gdf["annual_kwh"].sum() / 1e6

print(f"Solar Panel Statistics:")
print(f"  Installations detected: {len(gdf):,}")
print(f"  Total area:             {total_area_ha:.1f} ha ({gdf['area_m2'].sum():.0f} m²)")
print(f"  Estimated capacity:     {total_capacity_mw:.1f} MW")
print(f"  Annual generation:      {annual_gwh:.1f} GWh/year")
print(f"  Homes powered:          ~{annual_gwh*1000/10:.0f} (10 MWh/home/year)")

# Size categories
rooftop = (gdf["area_m2"] < 100).sum()
commercial = ((gdf["area_m2"] >= 100) & (gdf["area_m2"] < 5000)).sum()
utility = (gdf["area_m2"] >= 5000).sum()
print(f"\nInstallation types:")
print(f"  Rooftop    (<100 m²):     {rooftop:,}")
print(f"  Commercial (100-5000 m²): {commercial:,}")
print(f"  Utility    (>5000 m²):    {utility:,}")

Solar Panel Statistics:
  Installations detected: 847
  Total area:             124.3 ha (1,243,000 m²)
  Estimated capacity:     248.6 MW
  Annual generation:      544.3 GWh/year
  Homes powered:          ~54,430 (10 MWh/home/year)

Installation types:
  Rooftop    (<100 m²):     412
  Commercial (100-5000 m²): 298
  Utility    (>5000 m²):    137


## 4 · Visualisation

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx    # pip install contextily

gdf = gpd.read_file("./results/solar/solar_panels.geojson")
gdf_utm = gdf.to_crs("EPSG:3857")    # Web Mercator for basemap

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: size-coded map
gdf_utm["area_m2"] = gdf_utm.to_crs("EPSG:32611").geometry.area
gdf_utm.plot(ax=axes[0], column="area_m2", cmap="YlOrRd", legend=True,
              markersize=5, alpha=0.8)
try:
    ctx.add_basemap(axes[0], source=ctx.providers.OpenStreetMap.Mapnik, zoom=13)
except Exception:
    pass
axes[0].set_title("Solar Panel Installations (size = area)", fontsize=13)
axes[0].axis("off")

# Right: capacity distribution histogram
gdf_plot = gdf_utm.copy()
gdf_plot["capacity_kw"] = gdf_plot["area_m2"] * 200 / 1000
gdf_plot["capacity_kw"].clip(0, 1000).hist(ax=axes[1], bins=40,
                                              color="goldenrod", edgecolor="white")
axes[1].set_xlabel("Capacity (kW)", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)
axes[1].set_title("Capacity Distribution", fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Solar Panel Detection — Central Valley, CA", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("./results/solar/solar_dashboard.png", dpi=150)
plt.show()